# RAFT on Apache Jira

**Cases → extraction & review → BM25 + vectors → retrieval.** Use real Jira histories from the [paper's benchmark](../datasets/Apache_Jira/README.md), inspect a match, and measure case hit.

Run top to bottom with Python 3.11+. Install `pip install -e ".[notebook]" jupyterlab python-dotenv` from the repo root. This example loads `OPENAI_API_KEY` from your local `.env`; other clients/providers are configurable through the [Agents SDK](https://openai.github.io/openai-agents-python/config/). Live cells make billable calls.


In [ ]:
import json
import os
from pathlib import Path

from agents import Agent, ModelSettings, OpenAIResponsesModel, RunConfig
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import AsyncOpenAI

from raft import LocalPipeline
from raft.cases import restore_case
from raft.defaults import (
    REVIEWER_INSTRUCTIONS,
    WORKER_INSTRUCTIONS,
    CaseExtraction,
    CaseReview,
    case_to_text,
    state_to_text,
)
from raft.embedding.openai import OpenAIEmbeddings
from raft.storage import load_jsonl, save_json
from raft.tools import edit_state, query_case_sql

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'raft').is_dir())
load_dotenv(REPO / '.env', override=False)

CASE_LIMIT = 10  # Set None to index the full 600-case corpus.
MODEL = 'gpt-5.4'
EMBEDDING_MODEL = 'text-embedding-3-large'
CONCURRENCY = 20
RPM = 100
TOP_K = 5
BUILD_GRAPH = False
RUN_NAME = 'jira-walkthrough'  # Use a new name after changing models, prompts, or data.
OUTPUT_DIR = REPO / 'outputs' / RUN_NAME


## 1. Load the cases

The corpus contains 600 historical issues; the 30 duplicate reports are held out. `conversations` already combines the description and comments, so don't concatenate them again.

**Customize:** map your IDs, metadata, and ordered artifacts here. Keep labels such as `role`, `cluster`, and `target_key` out of model inputs and retrieval filters.


In [ ]:
data_dir = REPO / 'datasets' / 'Apache_Jira'
corpus = load_jsonl(data_dir / 'corpus.jsonl')
queries = load_jsonl(data_dir / 'queries.jsonl')
selected = corpus if CASE_LIMIT is None else corpus[:CASE_LIMIT]
cases = [
    {'id': row['key'],
     'metadata': {'source': row['metadata']['source'], 'project': row['project'],
                  'summary': row['summary']},
     'artifacts': row['conversations']}
    for row in selected
]
assert cases and len({case['id'] for case in cases}) == len(cases)
assert not {case['id'] for case in cases} & {query['key'] for query in queries}
print(f'{len(cases)} corpus cases selected; {len(queries)} held-out reports.')
print(cases[0]['id'], '-', cases[0]['metadata']['summary'])
print(f"{len(cases[0]['artifacts'])} ordered artifacts in this case.")


## 2. Define the worker and reviewer

The worker reads artifact batches and edits case state; the reviewer checks the result against evidence. Both can query the source, and the reviewer can inspect revision history.

**Customize:** replace the prompts or Pydantic output model, choose separate models for each role, or add domain tools. Keep `query_case_sql` and `edit_state` on both agents.


In [ ]:
client = AsyncOpenAI(api_key=os.environ['OPENAI_API_KEY'], max_retries=0, timeout=180)
model = OpenAIResponsesModel(model=MODEL, openai_client=client)
settings = ModelSettings(reasoning={'effort': 'low'}, max_tokens=8000, store=False)
tools = [query_case_sql, edit_state]

worker = Agent(name='Case worker', model=model, model_settings=settings,
               instructions=WORKER_INSTRUCTIONS, tools=tools)
reviewer = Agent(name='Case reviewer', model=model, model_settings=settings,
                 instructions=REVIEWER_INSTRUCTIONS, tools=tools, output_type=CaseReview)


## 3. Configure indexing

`LocalPipeline` saves extracted cases, timeline vectors, and a BM25 index. No graph is needed for hybrid retrieval.

**Customize:** `max_batch_chars` controls whole-artifact batches; `state_to_text` chooses embedding text. Swap the embedding backend, set `bm25=False` for vector-only retrieval, or change `should_keep`. Concurrency and RPM are tunable; your provider's token limit still applies.


In [ ]:
extraction = {
    'worker_agent': worker, 'reviewer_agent': reviewer, 'output_type': CaseExtraction,
    'id_field': 'id', 'metadata_field': 'metadata', 'artifacts_field': 'artifacts',
    'should_keep': lambda case: case.review.extractable,
    'max_batch_chars': 40_000, 'max_query_chars': 12_000,
    'concurrency': CONCURRENCY, 'agent_concurrency': CONCURRENCY, 'rpm': RPM,
    'timeout': 900, 'retries': 3, 'run_config': RunConfig(tracing_disabled=True),
}
embedding = {
    'backend': OpenAIEmbeddings(client=client, model=EMBEDDING_MODEL),
    'state_to_text': state_to_text, 'batch_size': 32,
    'concurrency': CONCURRENCY, 'rpm': RPM,
}
pipeline = LocalPipeline(OUTPUT_DIR, extraction=extraction, embedding=embedding,
                         bm25=True, graph=None, show_progress=True)


In [ ]:
indexed = await pipeline.index(cases)
save_json(OUTPUT_DIR / 'last_extraction.json', indexed['extraction'])
failures = indexed['extraction']['failed_cases'] + indexed['embedding']['failed_cases']
if failures:
    for failure in failures[:5]:
        print(failure['id'], failure['error_type'], failure['error_message'])
    raise RuntimeError('Indexing incomplete. Inspect saved failures and rerun this cell.')
print(json.dumps(indexed['summary'], indent=2))


## 4. Read one extracted case

Each narrative describes a meaningful investigation state, not a message-by-message summary. The case also retains its final review and worker/reviewer revisions.

**Customize:** edit [the defaults](../src/raft/defaults/) or supply your own schema. `handoff_notes` carries unfinished context between worker passes; unknown conclusions stay `None`.


In [ ]:
catalog = json.loads((OUTPUT_DIR / 'catalog.json').read_text(encoding='utf-8'))
stored_cases = [restore_case(record['case'], CaseExtraction)
                for record in catalog['cases'].values() if record['status'] == 'embedded']
if not stored_cases:
    raise RuntimeError('No searchable cases were indexed.')
sample = stored_cases[0]
print(sample.id, '-', sample.metadata['summary'])
for number, entry in enumerate(sample.output.timeline, 1):
    print(f'\n{number}. {entry.narrative}')
print('\nResolution:', sample.output.resolution_steps)
print('Worker passes:', sample.execution['passes'])


## 5. Retrieve similar cases

Query text is ranked against timeline entries; each hit returns its parent case and matched entry. This first search uses a held-out report, not an indexed case.

**Customize:** `top_k` caps distinct cases; `max_chars` caps the returned `formatted_context`, including separators. The default text contains ID, metadata, extracted state, and the matched `item_index`—not execution history. A custom `format_case(hit)` can use the full `hit['case']` and matched index to return your preferred text. `case_filter` can restrict results by project.


In [ ]:
question = next(query for query in queries if query['progress_valid']['0'])
print('Query:', question['key'])
print(question['query_0'][:500])
retrieved = await pipeline.retrieve([question['query_0']], top_k=TOP_K)
result = retrieved['results'][0]
if result['error']:
    raise RuntimeError(result['error'])
for hit in result['candidates']:
    case = hit['case']
    print(f"{case.id}: entry {hit['item_index'] + 1} - {case.metadata['summary']}")
if result['candidates']:
    best = result['candidates'][0]
    print('\nBest matching state:', best['case'].output.timeline[best['item_index']].narrative)
print('\nFormatted context characters:', result['used_chars'])


In [ ]:
filtered = await pipeline.retrieve(
    [question['query_0']], top_k=TOP_K, max_chars=16_000,
    format_case=lambda hit: hit['case'].output.model_dump_json(),
    case_filter=lambda query, case: case.metadata['project'] == question['project'],
)
if filtered['results'][0]['error']:
    raise RuntimeError(filtered['results'][0]['error'])
print('Same-project matches:', [hit['id'] for hit in filtered['results'][0]['candidates']])


## 6. Optional: expand through related cases

Set `BUILD_GRAPH=True` in setup to enable this cell. Graph construction reuses the extracted cases and adds case-level embeddings; it does not repeat extraction. The final evaluation remains direct retrieval only.

**Customize:** `case_to_text` chooses the linking view (root cause + resolution by default); `top_k` controls neighbors during graph construction. Expansion has its own per-case limit.


In [ ]:
if BUILD_GRAPH:
    graph = {'backend': embedding['backend'], 'case_to_text': case_to_text,
             'top_k': 5, 'concurrency': CONCURRENCY, 'rpm': RPM}
    with_graph = LocalPipeline(OUTPUT_DIR, extraction=extraction, embedding=embedding,
                               graph=graph, show_progress=True)
    graph_result = await with_graph.index()
    if graph_result['graph']['failed_cases']:
        raise RuntimeError(graph_result['graph']['failed_cases'])
    groups = await with_graph.graph_expansion([sample.id], per_case_top_k=3)
    print('Neighbors of', sample.id, ':', [n['id'] for n in groups[0]['neighbors']])
else:
    print('Graph expansion is off; BM25 + vector retrieval needs no graph.')


## 7. Reopen and update

Reuse the same directory and configuration to retrieve later. `index(cases)` skips saved IDs; pass new cases to extend the index or `rewrite=True` to replace specific cases. Use a new output directory when changing extraction prompts/schema or embedding models.


In [ ]:
reopened = LocalPipeline(OUTPUT_DIR, extraction=extraction, embedding=embedding)
unchanged = await reopened.index(cases)
assert len(unchanged['skipped_ids']) == len(cases)
print(f"Reopened index; skipped {len(unchanged['skipped_ids'])} existing cases.")
print('Saved results:', OUTPUT_DIR.relative_to(REPO))


## 8. Case Hit at 0% progress

Use the **30 valid `query_0` texts from `queries.jsonl`**: just the initial reports. A Case Hit means the exact `target_key` appears in the retrieved cases. No LLM judge, graph expansion, or project filter is used here.

**Customize:** change `TOP_K`. This example measures top-k hits without a context-token budget. Use `CASE_LIMIT=None` for the full 600-case corpus; a smaller limit keeps all 30 queries but may exclude their targets.


In [ ]:
valid = [query for query in queries if query['progress_valid']['0']]
batch = await reopened.retrieve([query['query_0'] for query in valid],
                                 top_k=TOP_K, concurrency=CONCURRENCY, rpm=RPM)
evaluation = []
for query, result in zip(valid, batch['results'], strict=True):
    if result['error']:
        raise RuntimeError(f"{query['key']}: {result['error']}")
    ids = [hit['id'] for hit in result['candidates']]
    evaluation.append({'query': query['key'], 'target': query['target_key'],
                       'retrieved': ids, 'hit': query['target_key'] in ids})

hits = sum(row['hit'] for row in evaluation)
searchable_ids = {case.id for case in stored_cases}
available_targets = sum(query['target_key'] in searchable_ids for query in valid)
save_json(OUTPUT_DIR / 'case_hits.json', {'progress': 0, 'indexed_cases': len(cases),
                                       'top_k': TOP_K, 'results': evaluation})
await client.close()
display(Markdown(f'| Corpus cases | Progress | Queries | Targets in index | Hits | Case Hit@{TOP_K} |\n'
                 f'|---:|---|---:|---:|---:|---:|\n'
                 f'| {len(cases)} | 0% | {len(valid)} | {available_targets} | {hits} | {hits / len(valid):.1%} |'))
